# Démo 1 : gérer les données avec Dataset et DataLoader

Objectif : voir comment PyTorch sert les données à un modèle. Un **Dataset** donne les exemples un par un ; un **DataLoader** les regroupe en minibatchs. On écrit d'abord un Dataset pour les mesures du thermomètre du jour 1, puis un Dataset qui lit des images rangées dans un dossier, avec leurs classes dans un fichier CSV.

**Données** : onze mesures d'un thermomètre dont on ne connaît pas la graduation, avec la température correspondante en degrés Celsius.

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- écrire un Dataset en trois méthodes, d'abord pour les mesures du thermomètre, puis pour des images rangées dans un dossier ;
- servir les exemples par minibatchs mélangés avec un DataLoader ;
- découper les images en entraînement et validation, et parcourir une époque complète.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Les mesures du thermomètre

**On repart des onze mesures du thermomètre, vues au jour 1 avec le premier neurone.**

Pour comprendre comment PyTorch sert les données, on commence par un jeu minuscule, qu'on peut afficher en
entier.

Le code :
- `lectures` : ce qu'affiche le thermomètre, une valeur par mesure ;
- `celsius` : la vraie température de chaque mesure ;
- `.shape` : le nombre de valeurs de chaque tenseur.

In [ ]:
import torch

lectures = torch.tensor([35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4])
celsius = torch.tensor([0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0])
print(lectures.shape, celsius.shape)

Deux tenseurs de 11 valeurs : une lecture et une température par mesure.

## 2. Un Dataset en trois méthodes

**Un Dataset est une classe qui sait donner ses exemples un par un.**

Pensez au catalogue d'une bibliothèque : il sait combien de livres elle contient et va chercher celui dont
on lui donne le numéro. PyTorch n'en demande pas plus à un Dataset. On l'écrit comme une classe, sur le même
modèle que les classes `nn.Module` du jour 1.

Le code :
- `class MesuresThermometre(Dataset)` : la classe hérite de `Dataset`, la classe de base de PyTorch ;
- `__init__` : range les deux tenseurs dans l'objet ;
- `__len__` : renvoie le nombre d'exemples ;
- `__getitem__(self, i)` : renvoie l'exemple numéro `i`, ici le couple (lecture, température).

In [ ]:
from torch.utils.data import Dataset


class MesuresThermometre(Dataset):
    def __init__(self, lectures, celsius):
        self.lectures = lectures
        self.celsius = celsius

    def __len__(self):
        return len(self.lectures)

    def __getitem__(self, i):
        return self.lectures[i], self.celsius[i]

Rien à afficher : la classe est définie. Elle ne contient encore aucune donnée ; c'est l'appel
`MesuresThermometre(lectures, celsius)` qui crée le Dataset.

**À retenir** : un Dataset, c'est `__len__` et `__getitem__` ; `__init__` prépare ce dont elles ont besoin.

## 3. Interroger le Dataset

**On crée le Dataset, puis on lui pose ses deux questions.**

Le code :
- `MesuresThermometre(lectures, celsius)` : crée le Dataset ;
- `len(mesures)` : Python appelle `__len__` ;
- `mesures[0]` : Python appelle `__getitem__(0)`, le premier exemple.

In [ ]:
mesures = MesuresThermometre(lectures, celsius)
print(len(mesures))
print(mesures[0])


`len` donne 11 exemples ; `mesures[0]` renvoie le couple (35.7, 0.5) : la première lecture et sa
température.

**À retenir** : on ne manipule plus deux tenseurs séparés, mais des exemples complets, une entrée avec sa réponse.

## 4. Un DataLoader : des minibatchs

**Le DataLoader sert les exemples par paquets, les minibatchs.**

Pensez à un croupier : à chaque partie, il bat les cartes, puis les distribue par paquets. Le DataLoader
fait de même avec les exemples du Dataset : il les mélange, les demande un par un, et les assemble en
paquets prêts pour le modèle.

Le code :
- `DataLoader(mesures, batch_size=4, shuffle=True)` : des paquets de 4 exemples, dans un ordre mélangé ;
- `torch.manual_seed(0)` : le même mélange à chaque exécution ;
- `for x, y in loader:` : un paquet à la fois, avec les lectures `x` et les températures `y`.

Prédisez : onze mesures par paquets de 4, combien de paquets, et de quelle taille ?

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(0)
loader = DataLoader(mesures, batch_size=4, shuffle=True)
for x, y in loader:
    print(x)
    print(x.shape)
    print(y)
    print(y.shape)

Trois paquets : deux de 4 mesures, puis un de 3. Onze mesures ne se coupent pas en paquets de 4 : le
dernier paquet prend ce qui reste.

**À retenir** : la première dimension d'un paquet est le nombre d'exemples qu'il contient.

**Essayez** : ajoutez `drop_last=True` au DataLoader ; le paquet incomplet disparaît.

## 5. Une nouvelle époque, un nouvel ordre

**À chaque passage sur les données, le DataLoader mélange à nouveau.**

Une **époque** est un passage complet sur toutes les données. Si le modèle voyait toujours les mêmes paquets
dans le même ordre, il pourrait s'adapter à cet ordre ; le mélange l'en empêche.

Le code :
- `for epoch in range(1, 3)` : deux époques ;
- `for x, y in loader` : on reparcourt le même DataLoader ;
- `print("  ", x)` : les lectures de chaque paquet.

In [ ]:
for epoch in range(1, 3):
    print("époque", epoch)
    for x, y in loader:
        print("  ", x)

Les mêmes onze lectures à chaque époque, mais ni dans le même ordre, ni dans les mêmes paquets : le premier
paquet commence par 68.4 à l'époque 1, par 56.3 à l'époque 2.

**Essayez** : passez `shuffle=False` ; les paquets suivent alors l'ordre des mesures, le même à chaque époque.

## 6. Télécharger Fashion-MNIST

**On passe à de vraies images : des vêtements en niveaux de gris.**

Le code :
- `datasets.FashionMNIST("data", train=True, download=True)` : télécharge les 60 000 images d'entraînement
  dans le dossier `data`, si elles n'y sont pas déjà ;
- `%%capture` : masque la barre de progression du téléchargement.

In [ ]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 7. Des images rangées dans un dossier 

**On fabrique des données comme on en trouve en entreprise : des fichiers image et un tableau des classes.**

Dans un vrai projet, les images arrivent rarement sous forme de tenseurs : ce sont des fichiers dans un
dossier, et leurs classes sont dans un fichier à part. On reproduit cette situation avec 1000 images de
Fashion-MNIST.

Le code (à survoler) :
- `datasets.FashionMNIST("data", train=True)` : sans transform, chaque exemple est une image au format PIL,
  le format d'image de Python ;
- `image.save(...)` : écrit chaque image dans `vetements/images/`, sous les noms `0000.png`, `0001.png`... ;
- `lignes` : une ligne par image, avec le nom du fichier et le numéro de sa classe ;
- `write_text(...)` : écrit ces lignes dans `vetements/etiquettes.csv`.

In [ ]:
from pathlib import Path

fashion = datasets.FashionMNIST("data", train=True)
dossier = Path("vetements")
(dossier / "images").mkdir(parents=True, exist_ok=True)
lignes = ["fichier,classe"]
for i in range(1000):
    image, classe = fashion[i]
    image.save(dossier / "images" / f"{i:04d}.png")
    lignes.append(f"{i:04d}.png,{classe}")
(dossier / "etiquettes.csv").write_text("\n".join(lignes))
print(len(lignes) - 1, "images écrites dans", dossier)

Le dossier `vetements` contient maintenant les 1000 images et le fichier `etiquettes.csv`. À partir d'ici, on
part seulement de ces fichiers, comme dans un projet réel.

## 8. Regarder les fichiers

**Avant d'écrire le Dataset, on regarde comment les données sont rangées.**

Le Dataset devra relier chaque image à sa classe : il faut savoir où sont les images et comment le fichier
CSV est écrit.

Le code :
- `sorted((dossier / "images").iterdir())` : la liste des fichiers du dossier, triée par nom ;
- `read_text().splitlines()[:4]` : les quatre premières lignes du fichier CSV.

In [ ]:
fichiers = sorted((dossier / "images").iterdir())
print(len(fichiers), "fichiers, par exemple", fichiers[0].name)
print((dossier / "etiquettes.csv").read_text().splitlines()[:4])

1000 fichiers PNG, nommés `0000.png`, `0001.png`, et ainsi de suite. Le CSV commence par une ligne d'en-tête,
`fichier,classe`, puis une ligne par image : `0000.png` est de la classe 9, `0001.png` et `0002.png` de la classe 0.

**À retenir** : la première ligne n'est pas un exemple ; le Dataset devra la sauter.

## 9. Un Dataset qui lit les fichiers

**Le même Dataset en trois méthodes, cette fois pour des fichiers image.**

Le Dataset ne charge pas toutes les images au départ : il garde seulement la liste des fichiers, et ouvre
une image quand on la lui demande. C'est ce qui permet de travailler avec des jeux trop gros pour la
mémoire.

Le code :
- `__init__` : lit le fichier CSV, saute la ligne d'en-tête avec `[1:]`, et garde pour chaque image le
  couple (nom du fichier, classe) dans `self.exemples` ;
- `transform` : une transformation à appliquer à chaque image lue, comme dans les jeux de torchvision ;
- `__getitem__` : ouvre le fichier avec `Image.open`, applique la transformation, et renvoie l'image et sa
  classe convertie en nombre avec `int`.

In [ ]:
from PIL import Image


class ImagesVetements(Dataset):
    def __init__(self, dossier, transform=None):
        self.dossier = Path(dossier)
        self.transform = transform
        lignes = (self.dossier / "etiquettes.csv").read_text().splitlines()[1:]
        self.exemples = [ligne.split(",") for ligne in lignes]

    def __len__(self):
        return len(self.exemples)

    def __getitem__(self, i):
        fichier, classe = self.exemples[i]
        image = Image.open(self.dossier / "images" / fichier)
        if self.transform is not None:
            image = self.transform(image)
        return image, int(classe)

Rien à afficher : la classe est définie. Aucune image n'est encore ouverte : `Image.open` ne s'exécute
que lorsqu'on demande un exemple.

**Piège fréquent** : oublier `transform=transforms.ToTensor()`. Le Dataset renvoie alors des images PIL, que le
DataLoader ne sait pas assembler en paquets : l'erreur n'apparaît qu'au premier paquet demandé.

## 10. Lire un exemple

**On crée le Dataset sur le dossier, et on lit le premier exemple.**

Le code :
- `ImagesVetements("vetements", transform=transforms.ToTensor())` : le Dataset, qui convertit chaque image
  lue en tenseur ;
- `vetements[0]` : ouvre `0000.png`, le convertit, et renvoie l'image et sa classe ;
- `fashion.classes[label]` : le nom de la classe correspondant au numéro.

Prédisez : quelle forme pour une image en niveaux de gris de 28 x 28 pixels ?

In [ ]:
from torchvision import transforms

vetements = ImagesVetements("vetements", transform=transforms.ToTensor())
image, label = vetements[3]
print(len(vetements), image.shape, label, fashion.classes[label])

1000 exemples. Le premier est un tenseur `[1, 28, 28]` : un canal de gris, 28 x 28 pixels. Sa classe est 9,
`Ankle boot`, une bottine, comme l'indiquait la première ligne du CSV.

**À retenir** : le Dataset a ouvert `0000.png` au moment où on l'a demandé, et la transform l'a converti en tenseur.

## 11. Quelques images, lues par le Dataset

**Huit images demandées au Dataset, pour vérifier qu'il renvoie la bonne classe avec la bonne image.**

Une erreur de lecture, par exemple un décalage d'une ligne dans le fichier CSV, ne provoque aucun message :
seul un coup d'oeil aux images la révèle.

Le code (à survoler) : une rangée de huit cases ; pour chacune, on demande l'exemple `i` au Dataset et on
affiche l'image avec le nom de sa classe.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    image, label = vetements[i]
    ax.imshow(image[0], cmap="gray")
    ax.set_title(fashion.classes[label], fontsize=8)
    ax.axis("off")
plt.show()

Chaque image porte la bonne classe : une bottine, des hauts, une robe, des pulls, une basket. Les classes
des trois premières images sont aussi celles des trois premières lignes du CSV : le Dataset relie bien chaque
fichier à sa ligne.

## 12. Compter les images par classe

**Les classes sont-elles représentées de façon équilibrée ?**

Sur un extrait de 1000 images, rien ne garantit le même nombre d'images par classe. Il faut le savoir avant
de juger un modèle.

Le code :
- `[int(classe) for fichier, classe in vetements.exemples]` : la classe de chaque image, lue dans la liste
  du Dataset ;
- `torch.bincount(...)` : combien de fois apparaît chaque numéro de classe.

In [ ]:
etiquettes = torch.tensor([int(classe) for fichier, classe in vetements.exemples])
print(torch.bincount(etiquettes))

Entre 86 images (les pulls, `Pullover`) et 115 (les baskets, `Sneaker`) par classe : un extrait de 1000 images
n'est pas parfaitement équilibré, mais aucune classe n'est rare.

**À retenir** : répondre au hasard donnerait environ une bonne réponse sur dix ; c'est le repère pour juger un
futur modèle.

## 13. Entraînement et validation

**On sépare les images en deux jeux : l'un pour apprendre, l'autre pour vérifier.**

Un modèle peut apprendre ses exemples par coeur. On le vérifie sur des images qu'il n'a jamais vues
pendant l'entraînement : le jeu de **validation**.

Le code :
- `random_split(vetements, [800, 200])` : coupe le Dataset au hasard en deux Datasets de 800 et 200 images ;
- `torch.manual_seed(0)` : le même découpage à chaque exécution.

In [ ]:
from torch.utils.data import random_split

torch.manual_seed(0)
train_ds, val_ds = random_split(vetements, [800, 200])
print(len(train_ds), len(val_ds))

800 images pour entraîner, 200 pour valider. `random_split` ne copie aucune image : chaque partie garde une
liste de numéros, et demande les images au Dataset d'origine.

## 14. Les DataLoader des images

**Un DataLoader par jeu : mélangé pour l'entraînement, dans l'ordre pour la validation.**

On mélange l'entraînement pour que le modèle ne voie pas toujours les mêmes paquets. La validation ne
sert qu'à mesurer : l'ordre n'y change rien.

Le code :
- `DataLoader(train_ds, batch_size=32, shuffle=True)` : des paquets de 32 images mélangées ;
- `DataLoader(val_ds, batch_size=100)` : des paquets de 100, sans mélange ;
- `next(iter(train_loader))` : le premier paquet, pour regarder sa forme.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=100)
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

Un paquet : `[32, 1, 28, 28]`, soit 32 images d'un canal de 28 x 28 pixels, et leurs 32 classes. Le
DataLoader a demandé 32 exemples au Dataset et les a empilés en un seul tenseur.

**À retenir** : la forme d'un paquet d'images est (paquet, canaux, hauteur, largeur).

## 15. Une époque, paquet après paquet

**On parcourt une époque complète, comme le fera la boucle d'entraînement.**

C'est la boucle d'entraînement du jour 1, sans le modèle : à chaque tour, un paquet d'images et leurs
classes.

Le code :
- `for images, labels in train_loader` : un paquet à la fois, jusqu'à la fin du jeu ;
- `nb_images += len(images)` : on compte les images servies ;
- `len(train_loader)` : le nombre de paquets d'une époque.

In [ ]:
nb_images = 0
for images, labels in train_loader:
    nb_images += len(images)
print(len(train_loader), "paquets,", nb_images, "images")

25 paquets et 800 images : 800 / 32 = 25, tous les paquets sont complets. Chaque image d'entraînement a été
servie une fois pendant l'époque.

**À retenir** : c'est exactement la boucle de l'entraînement ; il ne reste qu'à faire passer chaque paquet dans
le modèle.

## À retenir de cette démo

- Un Dataset répond à deux questions : combien y a-t-il d'exemples (`__len__`) et quel est l'exemple numéro `i` (`__getitem__`) ; il peut lire chaque exemple seulement quand on le lui demande.
- Un DataLoader demande les exemples au Dataset et les regroupe en minibatchs, mélangés à chaque époque avec `shuffle=True` ; le dernier paquet peut être incomplet.
- `for images, labels in train_loader` est la boucle de l'entraînement : il ne manque plus que le modèle, la loss et l'optimiseur.